# Aula 9 — GenAI + Análise

Nas aulas anteriores construímos uma estratégia completa, com backtest rigoroso, métricas e validação out-of-sample. O que falta? **Comunicar tudo isso em linguagem humana.**

Esta aula mostra como usar LLMs (especificamente a API do Claude) para acelerar a produção do relatório — sem abrir mão da qualidade técnica.

---

**Agenda:**
1. Como LLMs funcionam — intuição em 10 minutos
2. Onde GenAI se encaixa (e onde não se encaixa) no processo quant
3. Setup da API do Claude
4. Engenharia de prompt para análise quantitativa
5. Escrevendo seções do relatório com o modelo
6. Workflow completo: métricas → análise → draft do relatório

In [ ]:
# ── CONFIGURAÇÃO DO AMBIENTE ─────────────────────────────────────────────────
# Este notebook roda no VS Code (Jupyter local) E no Google Colab.
# Execute esta célula primeiro — ela detecta o ambiente e configura os caminhos.

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Google Colab ──────────────────────────────────────────────────────────
    print("Ambiente detectado: Google Colab")

    # Montar Google Drive para persistir os dados entre sessões
    from google.colab import drive
    drive.mount('/content/drive')

    # Instalar dependências não incluídas no Colab por padrão
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'yfinance', 'pyarrow'], check=False)

    # Clonar o repositório do curso (pula automaticamente se já existir)
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        print("Clonando repositório do curso...")
        subprocess.run([
            'git', 'clone',
            'https://github.com/fmaldonadoo/intensivao-quant-ai.git',
            REPO_DIR
        ])
        print("Repositório clonado com sucesso.")

    # Pasta de dados no Google Drive — persiste entre sessões do Colab
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")

else:
    # ── VS Code / Jupyter local ───────────────────────────────────────────────
    print("Ambiente detectado: VS Code / Jupyter local")

    # Sobe a árvore de diretórios até encontrar a raiz do repositório (.git)
    _p = os.path.abspath(os.getcwd())
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')):
            break
        _p = os.path.dirname(_p)

    DADOS_DIR = os.path.join(_p, 'dados')
    os.makedirs(DADOS_DIR, exist_ok=True)
    print(f"Dados serão salvos em: {DADOS_DIR}")


---
## 1. Como LLMs funcionam

### A tarefa fundamental

Um LLM (Large Language Model) faz uma coisa: **prever o próximo token**.

```
Entrada:  "O Sharpe ratio da estratégia foi de"
Saída:    P("1.23" | contexto) = 0.08
          P("0.95" | contexto) = 0.06
          P("2.1"  | contexto) = 0.04
          ...
```

Um **token** é aproximadamente uma sílaba ou pedaço de palavra. "backtest" → ["back", "test"].

### Attention: o que conecta tudo

O mecanismo de **Attention** (Vaswani et al., 2017 — "Attention is All You Need") permite ao modelo aprender quais tokens do contexto são relevantes para prever o próximo.

```
"A estratégia de momentum com Sharpe de 1.8 superou o benchmark por ___"
                              ↑              ↑                        ↑
                           relevante      relevante               prevendo

Attention peso alto para: "Sharpe", "1.8", "superou"
Attention peso baixo para: "A", "de", "com"
```

### Por que funcionam tão bem para texto analítico?

LLMs foram treinados em enormes corpora de texto financeiro e acadêmico. Eles aprenderam padrões como:

- "Sharpe > 1.5 em equities" → associado a "forte performance ajustada a risco"
- "max drawdown de 30%" → associado a "período de crise" ou "2008"
- "turnover mensal de 40%" → associado a "custos elevados de transação"

Isso não é raciocínio — é **compressão de padrões linguísticos em escala massiva**. Mas a consequência prática é que o modelo escreve análises financeiras coerentes e bem estruturadas.

### O que LLMs **não** fazem

| Tarefa | LLM | Por quê |
|--------|-----|---------||
| Calcular métricas precisas | ✗ | Aritmética confiável requer código |
| Acessar dados em tempo real | ✗ | Conhecimento tem data de corte |
| Garantir veracidade factual | ✗ | Hallucinations são reais |
| Escrever análise com dados reais | ✓ | Se você passar os dados |
| Estruturar argumentos | ✓ | Ponto forte |
| Explicar conceitos | ✓ | Ponto forte |
| Identificar inconsistências no texto | ✓ | Com contexto adequado |

---
## 2. Onde GenAI se encaixa no processo quant

```
Processo Quant completo:

  [Dados]  →  [Sinal]  →  [Alocação]  →  [Backtest]  →  [Análise]  →  [Relatório]
     ↑            ↑           ↑               ↑              ↑               ↑
    Python      Python      Python          Python         Você +          Você +
    yfinance    numpy       pandas          scipy          Claude          Claude
```

### O que delegamos ao Claude

**Análise e interpretação:**
- "Dado esse Sharpe e drawdown, o que isso significa para o risco da estratégia?"
- "Como justificar a escolha da janela 12-1 para um júri não técnico?"

**Redação estruturada:**
- Resumo executivo da estratégia
- Seção de metodologia para o relatório
- Interpretação de métricas para cada cenário

**Revisão e crítica:**
- "Revise esta seção e identifique afirmações que precisam de evidência"
- "Quais pontos fracos um avaliador experiente apontaria nesta metodologia?"

### O que **não** delegamos

- Os cálculos em si (Sharpe, drawdown, alpha)
- A validação dos dados
- As decisões de parâmetro (fundamentadas na literatura)
- A interpretação final — você assina o relatório

---
## 3. Setup da API do Claude

In [ ]:
# Instalar o SDK da Anthropic (se necessário)
# !pip install anthropic

import anthropic
import os
import json
import textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# A API key deve estar na variável de ambiente ANTHROPIC_API_KEY
# Configure com: export ANTHROPIC_API_KEY="sk-ant-..."
# ou via arquivo .env + python-dotenv
client = anthropic.Anthropic()  # lê ANTHROPIC_API_KEY automaticamente

print('SDK Anthropic configurado.')
print(f'Versão: {anthropic.__version__}')

In [ ]:
# Modelos disponíveis (maio 2025)
# claude-sonnet-4-6      → melhor custo-benefício para análise
# claude-opus-4-7        → mais capaz, para análises críticas
# claude-haiku-4-5-20251001 → mais rápido e barato, para tarefas simples

MODELO = 'claude-sonnet-4-6'  # usaremos este durante a aula

def chamar_claude(prompt, system=None, modelo=MODELO, max_tokens=1500):
    """Wrapper simples para chamadas ao Claude."""
    kwargs = {
        'model': modelo,
        'max_tokens': max_tokens,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    if system:
        kwargs['system'] = system
    
    response = client.messages.create(**kwargs)
    return response.content[0].text

# Teste básico
resposta = chamar_claude('Em uma frase, o que é o Sharpe Ratio?')
print('Resposta:', resposta)

---
## 4. Engenharia de prompt para análise quantitativa

A qualidade da saída depende criticamente da qualidade do prompt. Vamos ver os padrões mais úteis.

### Princípio 1: Contexto rico antes do pedido

O modelo não conhece sua estratégia. Você precisa passar:
- O que é a estratégia
- Quais as métricas calculadas
- Qual o período e universo
- Para qual audiência é o texto

**Prompt ruim:**
```
"Analise a minha estratégia de momentum."
```

**Prompt bom:**
```
"Analise a estratégia de momentum cross-sectional no IBOVESPA
 descrita abaixo. Métricas: CAGR 18.3%, Sharpe 1.42, Max Drawdown -22%,
 Alpha 7.2% a.a. vs BOVA11. Período: Jan/2015 a Dez/2024.
 Audiência: júri de concurso com background em finanças quantitativas.
 Escreva um parágrafo de análise de performance."
```

In [ ]:
# Carregando métricas reais calculadas nas aulas anteriores
ret_mensais  = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
ret_ibov     = ret_mensais['BOVA11.SA']
ret_v1       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_v1.parquet')).squeeze()
ret_v2       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_carteira_sinal_v2.parquet')).squeeze()
ret_wf       = pd.read_parquet(os.path.join(DADOS_DIR, 'retorno_walkforward_liquido.parquet')).squeeze()
pesos_v2     = pd.read_parquet(os.path.join(DADOS_DIR, 'pesos_sinal_v2.parquet'))

def calcular_metricas(retornos, benchmark=None, nome='Estratégia', rf_mensal=0.0):
    r = retornos.dropna()
    excesso = r - rf_mensal
    n = len(r)
    cagr = (1 + r).prod() ** (12 / n) - 1
    vol  = r.std() * np.sqrt(12)
    sharpe = (excesso.mean() / excesso.std()) * np.sqrt(12) if excesso.std() > 0 else np.nan
    downside = excesso[excesso < 0].std() * np.sqrt(12)
    sortino  = (excesso.mean() * 12 / downside) if downside > 0 else np.nan
    cum = (1 + r).cumprod()
    dd  = cum / cum.cummax() - 1
    mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd != 0 else np.nan
    alpha, beta = np.nan, np.nan
    if benchmark is not None:
        b = benchmark.reindex(r.index).dropna()
        r2 = r.reindex(b.index)
        if len(r2) > 10:
            beta, intercept, *_ = stats.linregress(b.values, r2.values)
            alpha = intercept * 12
    return dict(nome=nome, cagr=cagr, vol=vol, sharpe=sharpe, sortino=sortino,
                max_drawdown=mdd, calmar=calmar, alpha=alpha, beta=beta,
                n_meses=n, inicio=r.index[0], fim=r.index[-1])

# Calcular todas as métricas
m_v1   = calcular_metricas(ret_v1,  benchmark=ret_ibov, nome='Momentum v1 (sinal bruto)')
m_v2   = calcular_metricas(ret_v2,  benchmark=ret_ibov, nome='Momentum v2 (vol-adjusted)')
m_wf   = calcular_metricas(ret_wf,  benchmark=ret_ibov, nome='Walk-forward líquido')
m_ibov = calcular_metricas(ret_ibov, nome='IBOVESPA (BOVA11)')

turnover_medio = (pesos_v2.diff().abs().sum(axis=1) / 2).dropna().mean()

print('Métricas calculadas:')
for m in [m_v1, m_v2, m_wf, m_ibov]:
    print(f"  {m['nome']}: CAGR={m['cagr']:.1%}, Sharpe={m['sharpe']:.2f}, MDD={m['max_drawdown']:.1%}")

In [ ]:
# Princípio 2: System prompt para definir o papel e o tom

SYSTEM_ANALISTA_QUANT = """Você é um analista quantitativo sênior com 15 anos de experiência em 
gestão de portfólios e pesquisa de fatores. Você escreve relatórios técnicos precisos e concisos 
para gestores e júris de competições de finanças quantitativas. Seu estilo é direto, evita 
exageros, cita limitações metodológicas com honestidade, e usa terminologia técnica correta.
Escreva sempre em português brasileiro."""

# Serializar métricas para texto estruturado
def metricas_para_texto(m):
    return f"""
Estratégia: {m['nome']}
Período: {m['inicio'].strftime('%b/%Y')} a {m['fim'].strftime('%b/%Y')} ({m['n_meses']} meses)
CAGR: {m['cagr']:.1%}
Volatilidade anual: {m['vol']:.1%}
Sharpe Ratio: {m['sharpe']:.2f}
Sortino Ratio: {m['sortino']:.2f}
Máximo Drawdown: {m['max_drawdown']:.1%}
Calmar Ratio: {m['calmar']:.2f}
Alpha a.a. vs BOVA11: {m['alpha']:.1%}
Beta vs BOVA11: {m['beta']:.2f}""".strip()

# Prompt com contexto rico
prompt_analise = f"""Analise a performance das estratégias abaixo e escreva dois parágrafos:
1) O que os números dizem sobre a qualidade das estratégias
2) A diferença mais importante entre v1, v2 e walk-forward

===== ESTRATÉGIA V1 =====
{metricas_para_texto(m_v1)}

===== ESTRATÉGIA V2 (vol-adjusted) =====
{metricas_para_texto(m_v2)}

===== WALK-FORWARD LÍQUIDO DE CUSTOS (0.5% por turno) =====
{metricas_para_texto(m_wf)}

===== BENCHMARK: IBOVESPA =====
{metricas_para_texto(m_ibov)}

Turnover médio mensal da estratégia v2: {turnover_medio:.1%}
"""

analise_performance = chamar_claude(prompt_analise, system=SYSTEM_ANALISTA_QUANT, max_tokens=600)
print('=== ANÁLISE DE PERFORMANCE ===')
print(textwrap.fill(analise_performance, width=90))

### Princípio 3: Pedir formato específico

LLMs seguem instruções de formato muito bem. Sempre especifique:
- Quantos parágrafos
- Tom (técnico, executivo, didático)
- O que **não** incluir (evita divagações)
- Estrutura esperada (se quiser JSON, markdown, lista)

In [ ]:
# Exemplo: geração de JSON estruturado
prompt_json = f"""Dado o seguinte resumo de uma estratégia quantitativa, retorne um JSON com:
- "pontos_fortes": lista de até 3 pontos fortes observados nas métricas
- "pontos_fracos": lista de até 3 limitações ou riscos
- "recomendacoes": lista de até 2 melhorias sugeridas

Retorne SOMENTE o JSON, sem texto adicional.

Métricas da estratégia de momentum vol-adjusted (walk-forward, líquida de custos):
{metricas_para_texto(m_wf)}
Turnover médio: {turnover_medio:.1%}/mês
Número de ações no portfólio: top-10% do IBOVESPA (~7-8 ações)
Benchmark: BOVA11 (ETF IBOVESPA)
"""

resposta_json = chamar_claude(prompt_json, system=SYSTEM_ANALISTA_QUANT, max_tokens=600)

try:
    analise_estruturada = json.loads(resposta_json)
    print('=== ANÁLISE ESTRUTURADA (JSON) ===')
    print('\nPONTOS FORTES:')
    for p in analise_estruturada.get('pontos_fortes', []):
        print(f'  + {p}')
    print('\nPONTOS FRACOS / LIMITAÇÕES:')
    for p in analise_estruturada.get('pontos_fracos', []):
        print(f'  - {p}')
    print('\nRECOMENDAÇÕES:')
    for r in analise_estruturada.get('recomendacoes', []):
        print(f'  → {r}')
except json.JSONDecodeError:
    print('Resposta (texto):')
    print(resposta_json)

### Princípio 4: Chain of thought — "pense antes de responder"

Para análises complexas, pedir ao modelo que raciocine passo a passo melhora a qualidade da resposta final. Isso é especialmente útil quando você quer que o modelo **identifique contradições** ou **avalie premissas**.

In [ ]:
# Chain of thought: o modelo primeiro raciocina, depois conclui
prompt_cot = f"""Você é um avaliador experiente de estratégias quantitativas.

Antes de responder, pense em voz alta:
1) Que premissas essa estratégia faz que podem não se sustentar fora da amostra?
2) Que tipos de regime de mercado beneficiam ou prejudicam momentum?
3) Há inconsistências entre as métricas que merecem atenção?

Depois de raciocinar, conclua com uma avaliação final em 3-5 frases.

ESTRATÉGIA: Momentum cross-sectional no IBOVESPA
- Sinal: retorno acumulado 12-1 meses, normalizado pela vol. realizada de 63 dias
- Portfólio: top 10% das ações por sinal (long-only, equal weight)
- Rebalanceamento: mensal
{metricas_para_texto(m_wf)}
"""

resposta_cot = chamar_claude(prompt_cot, system=SYSTEM_ANALISTA_QUANT, max_tokens=800)
print('=== ANÁLISE COM CHAIN OF THOUGHT ===')
# Exibir com quebras de linha razoáveis
for linha in resposta_cot.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

---
## 5. Escrevendo seções do relatório

Vamos produzir as principais seções do relatório de entrega usando a API. A estrutura típica de um relatório de competição quant tem:

1. **Resumo Executivo** — 1 página, para quem tem 5 minutos
2. **Tese de Investimento** — por que momentum funciona
3. **Metodologia** — como construímos a estratégia
4. **Resultados e Métricas** — o que os números dizem
5. **Análise de Risco** — o que pode dar errado
6. **Conclusão** — o que recomendamos

In [ ]:
# Seção 1: Resumo Executivo

CONTEXTO_ESTRATEGIA = f"""
CONTEXTO:
- Competição: Desafio Quant AI da Itaú Asset Management 2025
- Equipe: estudantes de graduação em engenharia/ciências da computação
- Horizonte de investimento: médio prazo (mensal)
- Universo: ações do IBOVESPA (~77 tickers)
- Período de backtesting: 2015-2024 (10 anos)

ESTRATÉGIA:
- Fator: momentum cross-sectional
- Sinal v2: retorno acumulado 12-1 meses dividido pela vol. realizada de 63 dias
- Portfólio: long-only, top 10% por sinal, equal weight
- Rebalanceamento: mensal
- Validação: walk-forward (treino 48m, teste 12m)
- Custo modelado: 0.5% por turno sobre turnover realizado

MÉTRICAS WALK-FORWARD LÍQUIDAS:
{metricas_para_texto(m_wf)}
Turnover médio mensal: {turnover_medio:.1%}

BENCHMARK:
{metricas_para_texto(m_ibov)}
"""

prompt_exec = f"""{CONTEXTO_ESTRATEGIA}

Escreva o Resumo Executivo do relatório. Deve ter:
- 3-4 parágrafos
- Parágrafo 1: o que é a estratégia e qual a tese central
- Parágrafo 2: principais resultados (use os números reais acima)
- Parágrafo 3: rigor metodológico (mencione walk-forward e custos)
- Parágrafo 4: próximos passos / limitações reconhecidas
- Tom: técnico mas acessível a gestores de risco experientes
- NÃO invente métricas. Use SOMENTE os números fornecidos acima.
"""

resumo_executivo = chamar_claude(prompt_exec, system=SYSTEM_ANALISTA_QUANT, max_tokens=700)
print('=== RESUMO EXECUTIVO ===')
for linha in resumo_executivo.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

In [ ]:
# Seção 2: Metodologia

prompt_metodologia = f"""{CONTEXTO_ESTRATEGIA}

Escreva a seção de Metodologia do relatório. Estruture em subseções:

**2.1 Universo de Ativos**
Descreva a escolha do IBOVESPA como universo e os filtros de qualidade aplicados 
(cobertura mínima de 80%, filtro de liquidez).

**2.2 Construção do Sinal**
Explique o sinal 12-1 (citar Jegadeesh & Titman, 1993), o skip month, e a 
normalização pela volatilidade realizada de 63 dias. Inclua a fórmula em LaTeX.

**2.3 Construção do Portfólio**
Descreva o ranking cross-sectional, seleção do top 10%, equal weight, 
e rebalanceamento mensal.

**2.4 Validação**
Descreva o protocolo walk-forward e a modelagem de custos de transação.

Tom: acadêmico e preciso. Cada subseção deve ter 2-3 parágrafos.
NÃO invente referências além de Jegadeesh & Titman (1993) e DeMiguel et al. (2009).
"""

metodologia = chamar_claude(prompt_metodologia, system=SYSTEM_ANALISTA_QUANT, max_tokens=1200)
print('=== METODOLOGIA ===')
for linha in metodologia.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90) if not linha.startswith('**') else linha)
    else:
        print()

In [ ]:
# Seção 3: Análise de Risco

prompt_risco = f"""{CONTEXTO_ESTRATEGIA}

Escreva a seção de Análise de Risco. Comente sobre os seguintes riscos:

1. Risco de modelo: o que pode fazer o fator momentum deixar de funcionar?
   (crowding, regime shift, momentum crash pós-crise)

2. Risco de mercado: beta de {m_wf['beta']:.2f} implica exposição ao mercado brasileiro.
   Como isso se traduz em cenários extremos?

3. Risco de liquidez: com portfólio concentrado em 7-8 ações e turnover de {turnover_medio:.0%}/mês,
   qual o risco de impacto de mercado em ações menos líquidas do IBOV?

4. Survivorship bias: use o IBOVESPA constituintes atuais, que pode não representar
   fielmente o universo histórico disponível no início do período.

5. Risco regulatório: ações no Brasil têm regras específicas de curto e dividendos.
   Como isso afeta a estratégia long-only?

Para cada risco, mencione a magnitude estimada e mitigações adotadas ou possíveis.
Tom: honesto sobre limitações, sem ser alarmista.
"""

analise_risco = chamar_claude(prompt_risco, system=SYSTEM_ANALISTA_QUANT, max_tokens=900)
print('=== ANÁLISE DE RISCO ===')
for linha in analise_risco.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

---
## 6. Workflow completo: métricas → draft do relatório

Agora integramos tudo em uma pipeline completa que:
1. Coleta as métricas calculadas no Python
2. Formata o contexto para o LLM
3. Gera cada seção do relatório
4. Salva o draft em markdown

In [ ]:
def gerar_secao(titulo, instrucoes, contexto, max_tokens=800):
    """Gera uma seção do relatório com feedback visual."""
    print(f'Gerando: {titulo}...')
    prompt = f"{contexto}\n\n{instrucoes}"
    conteudo = chamar_claude(prompt, system=SYSTEM_ANALISTA_QUANT, max_tokens=max_tokens)
    return f"## {titulo}\n\n{conteudo}\n"

# Instrução para conclusão
instrucao_conclusao = """
Escreva a Conclusão do relatório. Deve:
- Retomar a tese central em 1 frase
- Afirmar os principais resultados quantitativos (use os números reais)
- Reconhecer as limitações mais importantes
- Propor 2-3 extensões naturais da pesquisa (ex: multi-fator, custo real, outros mercados)
- Finalizar com uma declaração sobre o potencial prático da estratégia
Máximo 3 parágrafos.
"""

conclusao = gerar_secao('Conclusão', instrucao_conclusao, CONTEXTO_ESTRATEGIA, max_tokens=500)
print('\n' + conclusao)

In [ ]:
# Montar o relatório completo e salvar

header_relatorio = f"""# Relatório de Estratégia Quantitativa
## Momentum Cross-Sectional no IBOVESPA

**Equipe:** ImpactUFSCar — Diretoria Quant  
**Competição:** Desafio Quant AI da Itaú Asset Management 2025  
**Data:** {pd.Timestamp.today().strftime('%B %Y')}  

---

"""

# Tabela de métricas formatada em markdown
def tabela_metricas_md(*dicts_metricas):
    linhas = []
    linhas.append('| Métrica | ' + ' | '.join(d['nome'] for d in dicts_metricas) + ' |')
    linhas.append('|---------|' + '|'.join('---------' for _ in dicts_metricas) + '|')
    campos = [
        ('CAGR', 'cagr', '{:.1%}'),
        ('Vol. Anual', 'vol', '{:.1%}'),
        ('Sharpe', 'sharpe', '{:.2f}'),
        ('Sortino', 'sortino', '{:.2f}'),
        ('Max Drawdown', 'max_drawdown', '{:.1%}'),
        ('Calmar', 'calmar', '{:.2f}'),
        ('Alpha a.a.', 'alpha', '{:.1%}'),
        ('Beta', 'beta', '{:.2f}'),
    ]
    for label, key, fmt in campos:
        vals = []
        for d in dicts_metricas:
            v = d.get(key, float('nan'))
            vals.append(fmt.format(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A')
        linhas.append(f'| {label} | ' + ' | '.join(vals) + ' |')
    return '\n'.join(linhas)

tabela_md = tabela_metricas_md(m_v1, m_v2, m_wf, m_ibov)

relatorio_completo = (
    header_relatorio
    + '## 1. Resumo Executivo\n\n' + resumo_executivo + '\n\n'
    + '## 2. Metodologia\n\n' + metodologia + '\n\n'
    + '## 3. Resultados\n\n'
    + '### 3.1 Métricas de Performance\n\n'
    + tabela_md + '\n\n'
    + '### 3.2 Análise de Performance\n\n' + analise_performance + '\n\n'
    + '## 4. Análise de Risco\n\n' + analise_risco + '\n\n'
    + conclusao
)

# Salvar draft
with open('relatorio_draft.md', 'w', encoding='utf-8') as f:
    f.write(relatorio_completo)

print(f'Draft salvo: relatorio_draft.md')
print(f'Tamanho: {len(relatorio_completo):,} caracteres')
print(f'Palavras estimadas: {len(relatorio_completo.split()):,}')

---
## 7. Usando o Claude para crítica e revisão

Uma das aplicações mais poderosas: pedir ao modelo para **criticar** o próprio relatório.

In [ ]:
# Revisão crítica: o modelo avalia o relatório como se fosse um júri

SYSTEM_JURI = """Você é um júri experiente de uma competição de finanças quantitativas,
composto por gestores da Itaú Asset Management. Você avalia criticamente relatórios de 
estratégias quantitativas, identificando:
- Afirmações não suportadas por evidências
- Limitações metodológicas não reconhecidas pelo autor
- Métricas mal interpretadas
- Argumentos circulares ou fracos
Seja direto e construtivo."""

# Lemos o resumo executivo gerado
prompt_revisao = f"""Leia o resumo executivo abaixo e responda:

1. Quais as 3 afirmações mais fortes e bem fundamentadas?
2. Quais as 2-3 lacunas ou fraquezas que um avaliador exigente questionaria?
3. O que está faltando que uma estratégia realmente pronta para produção teria?

RESUMO EXECUTIVO:
{resumo_executivo}

Métricas de referência (para checar consistência):
{metricas_para_texto(m_wf)}
"""

critica = chamar_claude(prompt_revisao, system=SYSTEM_JURI, max_tokens=700)
print('=== CRÍTICA DO JÚRI ===')
for linha in critica.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

In [ ]:
# Conversa multi-turno: refinar uma seção com base na crítica
# O SDK suporta histórico de mensagens diretamente

def conversa_refinamento(secao_original, critica_recebida):
    """Refina uma seção com base em crítica usando conversa multi-turno."""
    mensagens = [
        {'role': 'user', 'content': f'Aqui está o resumo executivo que escrevi:\n\n{secao_original}'},
        {'role': 'assistant', 'content': 'Entendido. Li o resumo executivo.'},
        {'role': 'user', 'content': f'Um avaliador fez estas críticas:\n\n{critica_recebida}\n\n'
                                    'Reescreva o resumo incorporando as melhorias mais importantes. '
                                    'Mantenha o mesmo tamanho e os números originais.'}
    ]
    
    response = client.messages.create(
        model=MODELO,
        max_tokens=700,
        system=SYSTEM_ANALISTA_QUANT,
        messages=mensagens
    )
    return response.content[0].text

resumo_refinado = conversa_refinamento(resumo_executivo, critica)
print('=== RESUMO EXECUTIVO REFINADO ===')
for linha in resumo_refinado.split('\n'):
    if linha.strip():
        print(textwrap.fill(linha, width=90))
    else:
        print()

---
## 8. Custo e limites da API

Antes de usar em produção, entenda os custos e limites:

In [ ]:
# Estimativa de custo da sessão desta aula
# Preços aproximados de maio/2025 (verifique em anthropic.com/pricing)

CUSTO_INPUT_POR_1M_TOKENS  = 3.00   # USD — claude-sonnet-4-6
CUSTO_OUTPUT_POR_1M_TOKENS = 15.00  # USD — claude-sonnet-4-6

# Estimativa: cada chamada desta aula usou ~500 tokens input + ~600 tokens output
# 8 chamadas no total
n_chamadas = 8
tokens_input_total  = n_chamadas * 700   # média conservadora
tokens_output_total = n_chamadas * 700

custo_usd = (tokens_input_total  / 1_000_000 * CUSTO_INPUT_POR_1M_TOKENS +
             tokens_output_total / 1_000_000 * CUSTO_OUTPUT_POR_1M_TOKENS)

print(f'Tokens usados (estimativa):')
print(f'  Input:  ~{tokens_input_total:,} tokens')
print(f'  Output: ~{tokens_output_total:,} tokens')
print(f'  Custo total estimado: USD ${custo_usd:.4f} (~R$ {custo_usd * 5:.2f})')
print()
print('Para o relatório completo (10-15 chamadas): USD $0.05-0.15 (~R$ 0.25-0.75)')
print()
print('Comparação de modelos para este caso de uso:')
print(f'  claude-haiku-4-5-20251001:  ~10x mais barato, qualidade ok para rascunhos')
print(f'  claude-sonnet-4-6: ← recomendado para análise quant')
print(f'  claude-opus-4-7:   ~5x mais caro que Sonnet, para casos críticos')

---
## 9. Boas práticas de GenAI em finanças quantitativas

### O que fazer

| Prática | Por quê |
|---------|----------|
| Passe os números explicitamente no prompt | Evita hallucination de métricas |
| Use system prompt para definir o papel | Mantém tom consistente entre chamadas |
| Peça revisão do próprio output | Identifica inconsistências antes de você |
| Salve os prompts que funcionaram | Reprodutibilidade e iteração |
| Verifique toda afirmação factual | O modelo erra datas, nomes, valores |

### O que não fazer

| Armadilha | Consequência |
|-----------|-------------|
| Pedir cálculos sem passar os dados | Inventa números plausíveis mas errados |
| Usar output sem revisar | Erros sutis passam despercebidos |
| Citar referências geradas pelo modelo | Artigos inventados são comuns |
| Delegar decisões de parâmetro | "Qual janela usar?" → resposta genérica sem fundamento |
| Usar em tempo real sem cache | Custo e latência são inaceitáveis para trading |

### O papel correto do GenAI no processo quant

```
  VOCÊ faz:                    CLAUDE faz:
  ─────────                    ──────────
  Calcular métricas        →   Interpretar os números
  Escolher parâmetros      →   Ajudar a articular a justificativa
  Validar o backtest       →   Identificar fraquezas no texto
  Assinar o relatório      →   Redigir o draft inicial
  Defender a metodologia   →   Antecipar perguntas do júri
```

In [ ]:
# Bônus: gerar perguntas prováveis do júri e preparar respostas

prompt_perguntas = f"""{CONTEXTO_ESTRATEGIA}

Você é um júri da Itaú Asset Management. Gere as 5 perguntas mais difíceis
que você faria para a equipe que desenvolveu esta estratégia.

Para cada pergunta, inclua também uma indicação de "o que o júri quer ouvir"
— ou seja, a resposta ideal que demonstra domínio do tema.

Formato:
P1: [pergunta desafiadora]
→ O júri quer ouvir: [resposta ideal em 2-3 frases]

P2: ...
"""

perguntas_juri = chamar_claude(prompt_perguntas, system=SYSTEM_JURI, max_tokens=900)
print('=== PERGUNTAS PROVÁVEIS DO JÚRI ===')
for linha in perguntas_juri.split('\n'):
    print(linha)

---
## Conclusão da Aula 9

LLMs não substituem o processo quantitativo — eles **aceleram a camada de comunicação**.

### O que aprendemos

1. **LLMs preveem tokens** — bons em linguagem, não em aritmética
2. **Contexto rico = saída rica** — passe os dados, não apenas o pedido
3. **System prompt define o papel** — analista sênior, júri, revisor
4. **Chain of thought melhora análises complexas** — peça raciocínio explícito
5. **Conversas multi-turno permitem refinamento** — itere com base em crítica
6. **Custo é acessível** — relatório completo por menos de R$ 1
7. **Revise sempre** — o modelo erra datas, nomes e inventar referências

### Arquivos gerados

- `relatorio_draft.md` — draft completo do relatório pronto para revisão humana

---
### Próxima aula

**Aula 10 — Relatório + Defesa:** como estruturar a apresentação final, defender escolhas metodológicas sob pressão, e conduzir a sessão de cross-review entre equipes.